In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from datetime import datetime
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Настройка графиков
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Цены на продовольствие (FAOSTAT)
df_prices = pd.read_csv('FAOSTAT_data_en_4-13-2026.csv', encoding='utf-8')
months_en = {'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
             'July':7,'August':8,'September':9,'October':10,'November':11,'December':12}

df_prices['month'] = df_prices['Months'].map(months_en)
df_prices['price'] = df_prices['Value'].astype(str).str.replace(',', '.').astype(float)

df = df_prices[['Year', 'month', 'price']].copy()
df.columns = ['year', 'month', 'price']
df = df.dropna()
df['date'] = pd.to_datetime(df['year'].astype(str) + '-' + df['month'].astype(str) + '-01')
df = df[(df['year'] >= 2018) & (df['year'] <= 2025)].sort_values('date').reset_index(drop=True)

# Ключевая ставка ЦБ
df_rate = pd.read_excel('Ключевая ставка ЦБ на конец месяца.xlsx',
                        sheet_name='Новый текстовый документ')
df_rate.columns = ['year', 'month', 'key_rate']
df_rate = df_rate.dropna()

# Инфляция
months_ru = {'янв':1,'фев':2,'мар':3,'апр':4,'май':5,'июн':6,
             'июл':7,'авг':8,'сен':9,'окт':10,'ноя':11,'дек':12}
df_inf_raw = pd.read_excel('Таблица инфляции по месяцам.xlsx', sheet_name='Таблица 2')

inf_data = []
for _, row in df_inf_raw.iterrows():
    try:
        year = int(row['Год'])
        if year < 2018 or year > 2025:
            continue
        for m_name, m_num in months_ru.items():
            val = row[m_name]
            if pd.notna(val):
                inf_data.append({'year': year, 'month': m_num,
                                'inflation': float(str(val).replace(',', '.'))})
    except:
        pass
df_inf = pd.DataFrame(inf_data)

# Курс доллара
df_usd_raw = pd.read_excel('Курсы доллара помесячно.xlsx', header=1)
usd_data = []
for _, row in df_usd_raw.iterrows():
    if pd.isna(row.iloc[0]):
        continue
    try:
        date_str = str(row.iloc[0]).split(' ')[0]
        date = datetime.strptime(date_str, '%Y-%m-%d')
        if date.year < 2018 or date.year > 2025:
            continue
        match = re.search(r'(\d+\.?\d*)', str(row.iloc[1]))
        if match:
            usd_data.append({'year': date.year, 'month': date.month,
                            'usd_rub': float(match.group(1))})
    except:
        pass
df_usd = pd.DataFrame(usd_data)

# Геополитический индекс GPR
df_gpr_raw = pd.read_excel('Геополитический индекс.xlsx', header=0)
gpr_data = []
for _, row in df_gpr_raw.iterrows():
    if pd.isna(row.iloc[0]):
        continue
    try:
        date = pd.to_datetime(row.iloc[0])
        if 2018 <= date.year <= 2025:
            gpr_data.append({'year': date.year, 'month': date.month,
                            'gpr': float(row.iloc[1])})
    except:
        pass
df_gpr = pd.DataFrame(gpr_data)

# Объединяем данные

df_all = df.merge(df_inf, on=['year','month'], how='left')
df_all = df_all.merge(df_rate, on=['year','month'], how='left')
df_all = df_all.merge(df_usd, on=['year','month'], how='left')
df_all = df_all.merge(df_gpr, on=['year','month'], how='left')
df_clean = df_all.dropna().reset_index(drop=True)

print(f"   Загружено: {len(df_clean)} месяцев ({df_clean['year'].min()}-{df_clean['year'].max()})")

# График 1: Динамика цен

plt.figure(figsize=(14, 6))
plt.plot(df_clean['date'], df_clean['price'], 'navy', linewidth=3, label='Индекс цен FAO')
plt.axvline(x=pd.to_datetime('2020-03-01'), color='darkorange', linestyle='--', linewidth=2, alpha=0.9, label='COVID-19')
plt.axvline(x=pd.to_datetime('2022-02-01'), color='red', linestyle='--', linewidth=2, alpha=0.9, label='Начало СВО')
plt.xlabel('Дата', fontsize=12)
plt.ylabel('Индекс цен (2015=100)', fontsize=12)
plt.title('Динамика мировых цен на продовольствие (FAO Food Price Index)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11, loc='upper left')
plt.grid(True, alpha=0.4, linewidth=0.8)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('price_dynamics.png', dpi=200)
plt.show()

# График 2: Общий

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# График 3: Цены
axes[0,0].plot(df_clean['date'], df_clean['price'], 'navy', linewidth=2.5)
axes[0,0].set_title('Цены на продовольствие (FAO, 2015=100)', fontsize=12, fontweight='bold')
axes[0,0].grid(True, alpha=0.4, linewidth=0.8)
axes[0,0].tick_params(axis='both', labelsize=10)

# График 4: Инфляция
colors = ['limegreen' if x >= 0 else 'crimson' for x in df_clean['inflation']]
axes[0,1].bar(df_clean['date'], df_clean['inflation'], color=colors, alpha=0.8, width=20)
axes[0,1].axhline(y=0, color='black', linewidth=1.5)
axes[0,1].set_title('Месячная инфляция в России (%)', fontsize=12, fontweight='bold')
axes[0,1].grid(True, alpha=0.4, linewidth=0.8, axis='y')
axes[0,1].tick_params(axis='both', labelsize=10)

# График 5: Курс USD и ставка ЦБ
ax2 = axes[1,0]
ax2.plot(df_clean['date'], df_clean['usd_rub'], 'darkblue', linewidth=2.5, label='Курс USD/RUB')
ax2.set_ylabel('Курс USD/RUB', color='darkblue', fontsize=11, fontweight='bold')
ax2.tick_params(axis='y', labelcolor='darkblue', labelsize=10)
ax3 = ax2.twinx()
ax3.plot(df_clean['date'], df_clean['key_rate'], 'darkred', linewidth=2.5, label='Ключевая ставка ЦБ')
ax3.set_ylabel('Ставка ЦБ (%)', color='darkred', fontsize=11, fontweight='bold')
ax3.tick_params(axis='y', labelcolor='darkred', labelsize=10)
ax2.set_title('Курс доллара и ключевая ставка ЦБ РФ', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.4, linewidth=0.8)
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax3.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)

# График 6: Геополитический риск
axes[1,1].plot(df_clean['date'], df_clean['gpr'], 'darkviolet', linewidth=2.5)
axes[1,1].fill_between(df_clean['date'], 0, df_clean['gpr'], alpha=0.4, color='darkviolet')
axes[1,1].axvline(x=pd.to_datetime('2022-02-01'), color='red', linestyle='--', linewidth=2, alpha=0.9, label='Начало СВО')
axes[1,1].set_title('Индекс геополитического риска (GPR)', fontsize=12, fontweight='bold')
axes[1,1].legend(loc='upper left', fontsize=10)
axes[1,1].grid(True, alpha=0.4, linewidth=0.8)
axes[1,1].tick_params(axis='both', labelsize=10)

plt.tight_layout()
plt.savefig('all_factors.png', dpi=200)
plt.show()


print("\n Корреляционный анализ")

corr = df_clean[['price', 'inflation', 'key_rate', 'usd_rub', 'gpr']].corr()

print("\n   Корреляция факторов с индексом цен:")
print(f"   - Инфляция:      {corr.loc['price', 'inflation']:.3f}")
print(f"   - Ставка ЦБ:     {corr.loc['price', 'key_rate']:.3f}")
print(f"   - Курс USD/RUB:  {corr.loc['price', 'usd_rub']:.3f}")
print(f"   - GPR:           {corr.loc['price', 'gpr']:.3f}")

# Тепловая карта
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f',
            square=True, linewidths=1.5, linecolor='white',
            annot_kws={'size': 13, 'fontweight': 'bold'},
            cbar_kws={'shrink': 0.8, 'label': 'Коэффициент корреляции'},
            xticklabels=['Цена', 'Инфляция', 'Ставка ЦБ', 'USD/RUB', 'GPR'],
            yticklabels=['Цена', 'Инфляция', 'Ставка ЦБ', 'USD/RUB', 'GPR'])
plt.title('Матрица корреляции факторов', fontsize=14, fontweight='bold', pad=20)
plt.xticks(fontsize=11, rotation=45)
plt.yticks(fontsize=11, rotation=0)
plt.tight_layout()
plt.savefig('corr_matrix.png', dpi=200)
plt.show()


print("\n Подготовка данных для моделирования")

df_model = df_clean.copy()

# Сезонность
df_model['month_sin'] = np.sin(2 * np.pi * df_model['month'] / 12)
df_model['month_cos'] = np.cos(2 * np.pi * df_model['month'] / 12)

# Лаги цен
df_model['price_lag1'] = df_model['price'].shift(1)
df_model['price_lag2'] = df_model['price'].shift(2)
df_model['price_lag3'] = df_model['price'].shift(3)

# Изменения макрофакторов
df_model['inf_change'] = df_model['inflation'].diff()
df_model['rate_change'] = df_model['key_rate'].diff()
df_model['usd_change'] = df_model['usd_rub'].pct_change() * 100
df_model['gpr_change'] = df_model['gpr'].pct_change() * 100

# Лаги изменений
df_model['inf_change_lag1'] = df_model['inf_change'].shift(1)
df_model['rate_change_lag1'] = df_model['rate_change'].shift(1)
df_model['usd_change_lag1'] = df_model['usd_change'].shift(1)
df_model['gpr_change_lag1'] = df_model['gpr_change'].shift(1)

df_model = df_model.dropna().reset_index(drop=True)
print(f"   Готово: {len(df_model)} наблюдений")

# Разделение на TRAIN/TEST

test_size = 12
train = df_model.iloc[:-test_size].copy()
test = df_model.iloc[-test_size:].copy()

print(f"\n Разделение данных:")
print(f"   Обучающая: {len(train)} мес.")
print(f"   Тестовая:  {len(test)} мес.")

y_train = train['price'].values
y_test = test['price'].values


print("\n Построение моделей:")

features_1 = ['price_lag1', 'price_lag2', 'price_lag3', 'month_sin', 'month_cos']

scaler1 = StandardScaler()
X_train1 = scaler1.fit_transform(train[features_1])
X_test1 = scaler1.transform(test[features_1])

model1 = Ridge(alpha=1.0)
model1.fit(X_train1, y_train)
pred1 = model1.predict(X_test1)

mae1 = mean_absolute_error(y_test, pred1)
r2_1 = r2_score(y_test, pred1)

print(f"\n   Модель 1 (авторегрессия):")
print(f"   MAE = {mae1:.3f}, R² = {r2_1:.3f}")

features_2 = features_1 + ['inf_change_lag1', 'rate_change_lag1', 'usd_change_lag1']

scaler2 = StandardScaler()
X_train2 = scaler2.fit_transform(train[features_2])
X_test2 = scaler2.transform(test[features_2])

model2 = Ridge(alpha=1.0)
model2.fit(X_train2, y_train)
pred2 = model2.predict(X_test2)

mae2 = mean_absolute_error(y_test, pred2)
r2_2 = r2_score(y_test, pred2)

impr2 = (1 - mae2/mae1) * 100

print(f"\n   Модель 2 (+ макрофакторы):")
print(f"   MAE = {mae2:.3f}, R² = {r2_2:.3f}")
print(f"   Улучшение: {impr2:+.1f}%")



features_3 = features_2 + ['gpr_change_lag1']

scaler3 = StandardScaler()
X_train3 = scaler3.fit_transform(train[features_3])
X_test3 = scaler3.transform(test[features_3])

model3 = Ridge(alpha=1.0)
model3.fit(X_train3, y_train)
pred3 = model3.predict(X_test3)

mae3 = mean_absolute_error(y_test, pred3)
r2_3 = r2_score(y_test, pred3)

impr3 = (1 - mae3/mae1) * 100

print(f"\n   Модель 3 (+ макро + GPR):")
print(f"   MAE = {mae3:.3f}, R² = {r2_3:.3f}")
print(f"   Улучшение: {impr3:+.1f}%")

# График 7: Сравнение прогнозов

plt.figure(figsize=(14, 7))

plt.plot(test['date'], y_test, 'k-o', linewidth=3, markersize=8,
         label='Фактические значения', markerfacecolor='black')
plt.plot(test['date'], pred1, 'darkorange', linestyle='--', marker='s', markersize=6, linewidth=2,
         label=f'Модель 1: авторегрессия (MAE={mae1:.2f})', alpha=0.9)
plt.plot(test['date'], pred2, 'royalblue', linestyle='-.', marker='d', markersize=6, linewidth=2,
         label=f'Модель 2: + макрофакторы (MAE={mae2:.2f})', alpha=0.9)
plt.plot(test['date'], pred3, 'forestgreen', linestyle='-', marker='^', markersize=7, linewidth=2.5,
         label=f'Модель 3: + макро + GPR (MAE={mae3:.2f})', alpha=0.9)

plt.axvline(x=test['date'].iloc[0], color='red', linestyle=':', linewidth=2.5, alpha=0.8, label='Начало прогноза')

plt.xlabel('Дата', fontsize=13, fontweight='bold')
plt.ylabel('Индекс цен (2015=100)', fontsize=13, fontweight='bold')
plt.title('Сравнение прогнозов моделей на 12 месяцев', fontsize=15, fontweight='bold')
plt.legend(loc='upper left', fontsize=11)
plt.grid(True, alpha=0.4, linewidth=0.8)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=200)
plt.show()

# График 8: Сравнение ошибок

models = ['Модель 1\n(авторегрессия)', 'Модель 2\n(+ макро)', 'Модель 3\n(+ макро + GPR)']
mae_values = [mae1, mae2, mae3]
bar_colors = ['gray', 'steelblue', 'forestgreen']

plt.figure(figsize=(10, 7))
bars = plt.bar(models, mae_values, color=bar_colors, alpha=0.85, width=0.6, edgecolor='black', linewidth=1.5)
plt.ylabel('MAE (средняя абсолютная ошибка)', fontsize=13, fontweight='bold')
plt.title('Сравнение точности моделей\n(меньше = лучше)', fontsize=15, fontweight='bold')

# Настройка оси Y с меньшим шагом для лучшей видимости разницы
y_min = min(mae_values) - 0.2
y_max = max(mae_values) + 0.3
plt.ylim(y_min, y_max)

# Добавляем больше делений на оси Y
plt.gca().yaxis.set_major_locator(plt.MultipleLocator(0.1))
plt.gca().yaxis.set_minor_locator(plt.MultipleLocator(0.05))

# Подписи значений над столбцами
for bar, val, color in zip(bars, mae_values, bar_colors):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.3f}', ha='center', va='bottom', fontsize=14, fontweight='bold', color=color)

# Подписи процентов улучшения
plt.text(1, mae1 + 0.08, 'базовый\nуровень', ha='center', fontsize=11, color='gray')
plt.text(2, mae2 + 0.08, f'{impr2:+.1f}%', ha='center', fontsize=12, fontweight='bold', color='steelblue')
plt.text(3, mae3 + 0.08, f'{impr3:+.1f}%', ha='center', fontsize=12, fontweight='bold', color='forestgreen')

plt.grid(True, alpha=0.3, axis='y', linewidth=0.8, which='both')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=200)
plt.show()

# Выволы

price_change = (df_clean['price'].iloc[-1] / df_clean['price'].iloc[0] - 1) * 100

print(f"""
1. Период исследования: 2018-2025 гг. ({len(df_clean)} месяцев)
2. Рост цен за весь период: {price_change:.1f}%

3. Корреляция факторов с индексом цен:
   - Ключевая ставка ЦБ:  {corr.loc['price', 'key_rate']:.3f}
   - Курс USD/RUB:        {corr.loc['price', 'usd_rub']:.3f}
   - Индекс GPR:          {corr.loc['price', 'gpr']:.3f}
   - Инфляция:            {corr.loc['price', 'inflation']:.3f}

4. Точность прогнозирования (MAE на 12 месяцев):
   - Модель 1 (авторегрессия):          {mae1:.3f}
   - Модель 2 (+ макрофакторы):         {mae2:.3f}  (улучшение {impr2:+.1f}%)
   - Модель 3 (+ макро + GPR):          {mae3:.3f}  (улучшение {impr3:+.1f}%)

5. Основные выводы:
   - Наибольшую корреляцию с ценами показывает ключевая ставка ЦБ (r = {corr.loc['price', 'key_rate']:.3f})
   - Учет макроэкономических факторов улучшает точность прогноза на {impr2:.1f}%
   - Геополитический фактор GPR имеет высокую корреляцию с ценами (r = {corr.loc['price', 'gpr']:.3f})
""")